In [ ]:
import re
from urllib.parse import unquote
from scapy.all import sniff, TCP, Raw

# Базовые паттерны XSS (без учета регистра)
XSS_PATTERNS = re.compile(
    r"(<script.*?>|on\w+\s*=|javascript:|alert\s*\(|document\.cookie)",
    re.IGNORECASE
)

def detect_xss(packet):
    if packet.haslayer(TCP) and packet.haslayer(Raw):
        # Фильтруем только HTTP-запросы (обычно порт назначения 80)
        if packet[TCP].dport == 80:
            raw_data = packet[Raw].load.decode(errors="ignore")
            # Декодируем URL-encoded символы (например, %3C -> <)
            decoded_data = unquote(raw_data)
            
            if XSS_PATTERNS.search(decoded_data):
                print(f"[XSS Alert] Source: {packet[0][1].src}")
                print(decoded_data[:300])  # Выводим только начало пакета
                print("-" * 40)

if __name__ == "__main__":
    print("Starting XSS detection on port 80...")
    sniff(filter="tcp dst port 80", prn=detect_xss, store=0)